# Обучение детектора промышленных деталей

**Подготовка:**
1. Загрузи папку `dataset/` в Google Drive как ZIP-архив `dataset.zip`
2. Укажи путь к архиву в ячейке «Настройки» ниже
3. Runtime → Change runtime type → **GPU**
4. Запускай ячейки сверху вниз

In [ ]:
# ── Настройки ─────────────────────────────────────────────────────────────
DATASET_ZIP   = '/content/drive/MyDrive/dataset.zip'  # путь к zip в Drive
SAVE_TO_DRIVE = '/content/drive/MyDrive/detector_weights'  # куда сохранять веса

EPOCHS     = 50
BATCH_SIZE = 16
LR         = 1e-3
NUM_CLASSES = 3

In [ ]:
# ── Монтируем Google Drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Распаковываем датасет ──────────────────────────────────────────────────
import zipfile, os

os.makedirs('/content/dataset', exist_ok=True)
with zipfile.ZipFile(DATASET_ZIP, 'r') as z:
    z.extractall('/content/dataset')

# Проверяем структуру
for split in ['train', 'val', 'test']:
    img_dir = f'/content/dataset/images/{split}'
    n = len([f for f in os.listdir(img_dir) if f.endswith('.png')]) if os.path.exists(img_dir) else 0
    print(f'{split}: {n} изображений')

In [ ]:
%%writefile /content/architecture.py
"""
Архитектура кастомного CNN-детектора промышленных деталей.

Одностадийный детектор на основе сетки 20x20.
Каждая ячейка предсказывает:
  - objectness: вероятность наличия объекта (1 значение)
  - bbox:        tx, ty — смещение центра внутри ячейки; tw, th — размеры
  - class:       логиты классов (C значений)

Декодирование bbox из ячейки (gi=строка, gj=столбец), G=20:
  x_c = (gj + sigmoid(tx)) / G
  y_c = (gi + sigmoid(ty)) / G
  w   = sigmoid(tw)
  h   = sigmoid(th)
"""

import torch
import torch.nn as nn


class ConvBNReLU(nn.Module):
    """Базовый строительный блок: Conv2d -> BatchNorm2d -> ReLU."""

    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size, stride, padding, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class ResidualBlock(nn.Module):
    """
    Остаточный блок: два слоя ConvBNReLU + skip connection.
    Input/Output: (B, C, H, W) -> (B, C, H, W)
    """

    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            ConvBNReLU(channels, channels, 3, 1, 1),
            ConvBNReLU(channels, channels, 3, 1, 1),
        )

    def forward(self, x):
        return x + self.block(x)


class Backbone(nn.Module):
    """
    Backbone: последовательное уменьшение разрешения stride=2 свертками.
    Input:  (B,   3, 640, 640)
    Output: (B, 512,  20,  20)
    """

    def __init__(self):
        super().__init__()
        self.stage1 = nn.Sequential(
            ConvBNReLU(3, 32, 3, 2, 1),    # (B, 32, 320, 320)
            ConvBNReLU(32, 32, 3, 1, 1),
        )
        self.stage2 = nn.Sequential(
            ConvBNReLU(32, 64, 3, 2, 1),   # (B, 64, 160, 160)
            ResidualBlock(64),
        )
        self.stage3 = nn.Sequential(
            ConvBNReLU(64, 128, 3, 2, 1),  # (B, 128, 80, 80)
            ResidualBlock(128),
            ResidualBlock(128),
        )
        self.stage4 = nn.Sequential(
            ConvBNReLU(128, 256, 3, 2, 1), # (B, 256, 40, 40)
            ResidualBlock(256),
            ResidualBlock(256),
        )
        self.stage5 = nn.Sequential(
            ConvBNReLU(256, 512, 3, 2, 1), # (B, 512, 20, 20)
            ResidualBlock(512),
        )

    def forward(self, x):
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        x = self.stage5(x)
        return x


class DetectionHead(nn.Module):
    """
    Голова детекции.
    Input:  (B, 512, 20, 20)
    Output: (B, 1+4+C, 20, 20)
    """

    def __init__(self, in_ch=512, num_classes=3):
        super().__init__()
        out_ch = 1 + 4 + num_classes
        self.head = nn.Sequential(
            ConvBNReLU(in_ch, 256, 3, 1, 1),
            ConvBNReLU(256, 128, 1, 1, 0),
            nn.Conv2d(128, out_ch, 1, 1, 0),
        )

    def forward(self, x):
        return self.head(x)


class Detector(nn.Module):
    """
    Полная модель детектора: Backbone + DetectionHead.
    Input:  (B, 3, 640, 640)
    Output: (B, 1+4+C, 20, 20)
    """

    GRID_SIZE = 20

    def __init__(self, num_classes=3):
        super().__init__()
        self.num_classes = num_classes
        self.backbone = Backbone()
        self.head = DetectionHead(in_ch=512, num_classes=num_classes)

    def forward(self, x):
        return self.head(self.backbone(x))

    def predict(self, x, conf_thresh=0.5):
        self.eval()
        with torch.no_grad():
            raw = self.forward(x)
        G = self.GRID_SIZE
        results = []
        for b in range(x.shape[0]):
            pred = raw[b]
            obj_map = torch.sigmoid(pred[0])
            max_conf = obj_map.max().item()
            if max_conf < conf_thresh:
                results.append(None)
                continue
            flat_idx = obj_map.argmax()
            gi = (flat_idx // G).item()
            gj = (flat_idx %  G).item()
            tx = torch.sigmoid(pred[1, gi, gj]).item()
            ty = torch.sigmoid(pred[2, gi, gj]).item()
            tw = torch.sigmoid(pred[3, gi, gj]).item()
            th = torch.sigmoid(pred[4, gi, gj]).item()
            results.append({
                'class': int(pred[5:, gi, gj].argmax().item()),
                'conf':  float(max_conf),
                'bbox':  [(gj + tx) / G, (gi + ty) / G, tw, th],
            })
        return results

In [ ]:
%%writefile /content/loss.py
"""
Функция потерь детектора.
Компоненты: objectness BCE + bbox SmoothL1 + class CrossEntropy.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class DetectionLoss(nn.Module):
    def __init__(self, grid_size=20, num_classes=3,
                 lambda_obj=1.0, lambda_noobj=0.5,
                 lambda_bbox=5.0, lambda_cls=1.0):
        super().__init__()
        self.G            = grid_size
        self.num_classes  = num_classes
        self.lambda_obj   = lambda_obj
        self.lambda_noobj = lambda_noobj
        self.lambda_bbox  = lambda_bbox
        self.lambda_cls   = lambda_cls

    def forward(self, predictions, targets):
        B, _, G, _ = predictions.shape
        device = predictions.device

        obj_mask = torch.zeros(B, G, G, device=device)
        tx_tgt   = torch.zeros(B, G, G, device=device)
        ty_tgt   = torch.zeros(B, G, G, device=device)
        tw_tgt   = torch.zeros(B, G, G, device=device)
        th_tgt   = torch.zeros(B, G, G, device=device)
        cls_tgt  = torch.zeros(B, G, G, dtype=torch.long, device=device)

        for b, tgt in enumerate(targets):
            if tgt is None:
                continue
            x_c, y_c, w, h = tgt['bbox']
            cls = int(tgt['class'])
            gj = min(int(x_c * G), G - 1)
            gi = min(int(y_c * G), G - 1)
            obj_mask[b, gi, gj] = 1.0
            tx_tgt[b, gi, gj]   = x_c * G - gj
            ty_tgt[b, gi, gj]   = y_c * G - gi
            tw_tgt[b, gi, gj]   = w
            th_tgt[b, gi, gj]   = h
            cls_tgt[b, gi, gj]  = cls

        noobj_mask = 1.0 - obj_mask
        pred_obj   = torch.sigmoid(predictions[:, 0, :, :])
        bce        = F.binary_cross_entropy(pred_obj, obj_mask, reduction='none')
        obj_loss   = (self.lambda_obj * (bce * obj_mask).sum() +
                      self.lambda_noobj * (bce * noobj_mask).sum()) / B

        pred_bbox  = torch.sigmoid(predictions[:, 1:5, :, :])
        bbox_tgt   = torch.stack([tx_tgt, ty_tgt, tw_tgt, th_tgt], dim=1)
        bbox_loss_map = F.smooth_l1_loss(pred_bbox, bbox_tgt, reduction='none').sum(dim=1)
        n_obj      = obj_mask.sum().clamp(min=1.0)
        bbox_loss  = self.lambda_bbox * (bbox_loss_map * obj_mask).sum() / n_obj

        pred_cls   = predictions[:, 5:, :, :]
        pos_indices = obj_mask.bool()
        if pos_indices.any():
            pred_pos = pred_cls.permute(0, 2, 3, 1)[pos_indices]
            cls_pos  = cls_tgt[pos_indices]
            cls_loss = self.lambda_cls * F.cross_entropy(pred_pos, cls_pos)
        else:
            cls_loss = torch.tensor(0.0, device=device)

        total = obj_loss + bbox_loss + cls_loss
        return total, {'obj': obj_loss.item(), 'bbox': bbox_loss.item(), 'cls': cls_loss.item()}

In [ ]:
%%writefile /content/train_colab.py
"""
Скрипт обучения для Google Colab.
Пути фиксированы под /content/.
"""

import csv
import os
import sys
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

sys.path.insert(0, '/content')
from architecture import Detector
from loss import DetectionLoss


class PartDataset(Dataset):
    def __init__(self, img_dir, lbl_dir, img_size=640):
        self.img_size  = img_size
        self.img_paths = sorted(Path(img_dir).glob('*.png'))
        self.lbl_dir   = Path(lbl_dir)
        self.img_paths = [
            p for p in self.img_paths
            if (self.lbl_dir / (p.stem + '.txt')).exists()
        ]

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        p = self.img_paths[idx]
        img = cv2.imread(str(p))
        img = cv2.resize(img, (self.img_size, self.img_size))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_t = torch.from_numpy(img.astype(np.float32) / 255.0).permute(2, 0, 1)
        with open(self.lbl_dir / (p.stem + '.txt')) as f:
            line = f.readline().strip().split()
        cls = int(line[0])
        x_c, y_c, w, h = float(line[1]), float(line[2]), float(line[3]), float(line[4])
        return img_t, {'class': cls, 'bbox': [x_c, y_c, w, h]}


def collate_fn(batch):
    imgs, targets = zip(*batch)
    return torch.stack(imgs), list(targets)


def train(dataset_root, epochs, batch_size, lr, device_str, save_dir, num_classes):
    device   = torch.device(device_str)
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    root     = Path(dataset_root)

    train_ds = PartDataset(root / 'images/train', root / 'labels/train')
    val_ds   = PartDataset(root / 'images/val',   root / 'labels/val')
    print(f'Train: {len(train_ds)}  Val: {len(val_ds)}')

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=2, collate_fn=collate_fn, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                              num_workers=2, collate_fn=collate_fn, pin_memory=True)

    model     = Detector(num_classes=num_classes).to(device)
    criterion = DetectionLoss(num_classes=num_classes)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5)

    log_path = save_dir / 'train_log.csv'
    with open(log_path, 'w', newline='') as f:
        csv.writer(f).writerow(['epoch', 'train_loss', 'val_loss', 'lr'])

    best_val = float('inf')

    for epoch in range(1, epochs + 1):
        model.train()
        train_sum = 0.0
        for i, (imgs, targets) in enumerate(train_loader):
            imgs = imgs.to(device)
            optimizer.zero_grad()
            loss, comp = criterion(model(imgs), targets)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 10.0)
            optimizer.step()
            train_sum += loss.item()
            if (i + 1) % 10 == 0:
                print(f'  [{epoch}/{epochs}][{i+1}/{len(train_loader)}] '
                      f'loss={loss.item():.4f} '
                      f'obj={comp["obj"]:.3f} bbox={comp["bbox"]:.3f} cls={comp["cls"]:.3f}')

        model.eval()
        val_sum = 0.0
        with torch.no_grad():
            for imgs, targets in val_loader:
                loss, _ = criterion(model(imgs.to(device)), targets)
                val_sum += loss.item()

        tr = train_sum / len(train_loader)
        vl = val_sum   / len(val_loader)
        cur_lr = optimizer.param_groups[0]['lr']
        print(f'Эпоха {epoch}: train={tr:.4f}  val={vl:.4f}  lr={cur_lr:.2e}')

        with open(log_path, 'a', newline='') as f:
            csv.writer(f).writerow([epoch, tr, vl, cur_lr])

        if vl < best_val:
            best_val = vl
            torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                        'val_loss': vl}, save_dir / 'detector_best.pt')
            print(f'  -> Сохранён checkpoint (val={vl:.4f})')

        scheduler.step(vl)

    print(f'Готово. Лучший val_loss={best_val:.4f}')
    return best_val

In [ ]:
# ── Проверка GPU ───────────────────────────────────────────────────────────
import torch
print('CUDA доступна:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    DEVICE = 'cuda'
else:
    print('GPU не найдена — обучение на CPU (медленно)')
    DEVICE = 'cpu'

In [ ]:
# ── Запуск обучения ────────────────────────────────────────────────────────
import sys
sys.path.insert(0, '/content')
from train_colab import train

WEIGHTS_DIR = '/content/weights'

train(
    dataset_root = '/content/dataset',
    epochs       = EPOCHS,
    batch_size   = BATCH_SIZE,
    lr           = LR,
    device_str   = DEVICE,
    save_dir     = WEIGHTS_DIR,
    num_classes  = NUM_CLASSES,
)

In [ ]:
# ── Копируем веса в Google Drive ───────────────────────────────────────────
import shutil, os

os.makedirs(SAVE_TO_DRIVE, exist_ok=True)
shutil.copy('/content/weights/detector_best.pt', SAVE_TO_DRIVE + '/detector_best.pt')
shutil.copy('/content/weights/train_log.csv',    SAVE_TO_DRIVE + '/train_log.csv')
print('Сохранено в Drive:', SAVE_TO_DRIVE)

In [ ]:
# ── График loss ────────────────────────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt

log = pd.read_csv('/content/weights/train_log.csv')
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(log['epoch'], log['train_loss'], label='Train loss')
ax.plot(log['epoch'], log['val_loss'],   label='Val loss')
ax.set_xlabel('Эпоха')
ax.set_ylabel('Loss')
ax.set_title('Кривые обучения детектора')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.savefig(SAVE_TO_DRIVE + '/loss_curve.png', dpi=150)
plt.show()